# Final XGBoost feature selection

This notebook tests conservative feature pruning using the existing XGBoost feature importance, SHAP, correlation, and VIF diagnostics. The original M1–M4 models remain unchanged.

Two candidates are retrained:

- **Accuracy-first:** M1 without `month`.
- **Research-preserving:** M4 without `rainfall_hourly_mm` and `indoor_humidity_pct_last_week`, while retaining representative indoor and weather variables.

A feature is recommended for removal only when the retrained candidate is no worse on the common scored operating-hours test rows.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
if ROOT.name!='4-XGBoost':
    ROOT=next(p for p in [ROOT,*ROOT.parents] if p.name=='4-XGBoost')
OUTPUT=ROOT/'outputs'
FINAL=OUTPUT/'final_models'
with open(FINAL/'feature_definitions.json') as f:
    definitions=json.load(f)
for name, details in definitions['definitions'].items():
    print(name, '| base:', details['base'], '| dropped:', ', '.join(details['dropped']))
    print('retained:', ', '.join(definitions['candidates'][name]))
    print()

accuracy_first | base: M1 | dropped: month
retained: floor, zone_code, hour, weekday, is_weekend, is_holiday, ac_load_kw_yesterday, ac_load_kw_2_days_ago, ac_load_kw_last_week

research_preserving | base: M4 | dropped: rainfall_hourly_mm, indoor_humidity_pct_last_week
retained: floor, zone_code, hour, weekday, month, is_weekend, is_holiday, ac_load_kw_yesterday, ac_load_kw_2_days_ago, ac_load_kw_last_week, indoor_temp_c_yesterday, indoor_temp_c_last_week, indoor_temp_c_yesterday_avg, indoor_humidity_pct_yesterday, indoor_humidity_pct_yesterday_avg, outdoor_temp_hourly_c, outdoor_temp_daily_avg_c, outdoor_humidity_hourly_pct, outdoor_humidity_daily_avg_pct, rainfall_daily_total_mm, solar_radiation_hourly_wm2, solar_radiation_daily_total_mjm2



## 1. Why these variables were considered

The existing M4 gain ranking showed low gain for hourly rainfall, several humidity lags, and some hourly weather variables. The multicollinearity diagnostic also showed shared information among floor/zone, outdoor temperature/humidity, and humidity summaries. Because correlated variables can split importance, only a small conservative set was tested.

In [2]:
importance=pd.concat([
    pd.read_csv(FINAL/'accuracy_first/feature_importance.csv').assign(model='accuracy_first'),
    pd.read_csv(FINAL/'research_preserving/feature_importance.csv').assign(model='research_preserving')
],ignore_index=True)
for model in importance.model.unique():
    print(model)
    display(importance[importance.model.eq(model)].sort_values('gain',ascending=False).head(15))

accuracy_first


,feature,gain,model
0,ac_load_kw_last_week,12615.795898,accuracy_first
1,is_holiday,11758.485352,accuracy_first
2,is_weekend,4035.730225,accuracy_first
3,ac_load_kw_yesterday,3250.302246,accuracy_first
4,weekday,1464.544922,accuracy_first
5,hour,1317.722412,accuracy_first
6,zone_code,288.546478,accuracy_first
7,ac_load_kw_2_days_ago,269.744080,accuracy_first
8,floor,213.256714,accuracy_first


research_preserving


,feature,gain,model
9,ac_load_kw_last_week,7148.612305,research_preserving
10,is_holiday,6230.617188,research_preserving
11,is_weekend,2680.698975,research_preserving
12,ac_load_kw_yesterday,2251.515625,research_preserving
13,weekday,1285.102173,research_preserving
14,solar_radiation_hourly_wm2,372.612122,research_preserving
15,indoor_temp_c_last_week,313.802307,research_preserving
16,hour,309.677094,research_preserving
17,zone_code,222.132751,research_preserving
18,indoor_temp_c_yesterday_avg,169.344147,research_preserving


## 2. Common scored test metrics

These results use the same `zone_id + timestamp` rows where the baseline file has `scored == 1`. Lower values are better.

In [3]:
reduced=pd.read_csv(FINAL/'common_metrics.csv').rename(columns={'slice':'slice'})
reduced['slice']=reduced['slice'].replace({'operating_hours':'operating hours','all_hours':'all hours'})
original=pd.read_csv(OUTPUT/'model_comparison/common_model_metrics.csv').rename(columns={'hours':'slice'})
comparison=pd.concat([
    original[['model','slice','n','MAE','RMSE','CV(RMSE)%','NMBE%','WAPE%']],
    reduced.rename(columns={'mae':'MAE','rmse':'RMSE','cv_rmse':'CV(RMSE)%','nmbe':'NMBE%','wape':'WAPE%'})[['model','slice','n','MAE','RMSE','CV(RMSE)%','NMBE%','WAPE%']]
],ignore_index=True)
display(comparison[comparison.slice.eq('operating_hours')].round(4))

,model,slice,n,MAE,RMSE,CV(RMSE)%,NMBE%,WAPE%


## 3. Recommendation

The accuracy-first reduced model is the recommended final predictive model because removing `month` improved every main operating-hours metric relative to the original XGBoost M1 in this run. The research-preserving reduction is useful for interpretation, but its performance is worse than both M1 and M3, so it should not replace the accuracy-first model for deployment.

In [4]:
operating=comparison[comparison.slice.eq('operating hours')].set_index('model')
base=operating.loc['XGBoost M1']
for model in ['accuracy_first','research_preserving']:
    row=operating.loc[model]
    print(model)
    for metric in ['MAE','RMSE','CV(RMSE)%','WAPE%']:
        print(f'  {metric}: {row[metric]:.4f} (change vs M1: {row[metric]-base[metric]:+.4f})')
print('\nRecommended final model: accuracy_first')
print('Recommended drop: month')
print('Do not drop floor based on VIF alone: the earlier floor ablation worsened M1 performance.')

accuracy_first
  MAE: 1.4788 (change vs M1: -0.0207)
  RMSE: 2.4670 (change vs M1: -0.0150)
  CV(RMSE)%: 0.1804 (change vs M1: -17.9707)
  WAPE%: 0.1081 (change vs M1: -10.8580)
research_preserving
  MAE: 1.7598 (change vs M1: +0.2602)
  RMSE: 2.7441 (change vs M1: +0.2621)
  CV(RMSE)%: 0.2007 (change vs M1: -17.9504)
  WAPE%: 0.1287 (change vs M1: -10.8374)

Recommended final model: accuracy_first
Recommended drop: month
Do not drop floor based on VIF alone: the earlier floor ablation worsened M1 performance.


## 4. Retained and dropped variables

### Accuracy-first final model

Drop only `month` from M1. Retain AC lags, operating/calendar variables, `zone_code`, and `floor`. This model achieved operating-hours WAPE of approximately **10.81%**, compared with **10.97%** for the original M1.

### Research-preserving model

Drop `rainfall_hourly_mm` and `indoor_humidity_pct_last_week`, but retain the remaining indoor and weather groups. This preserves the research comparison but achieved operating-hours WAPE of approximately **12.87%**, so it is not the recommended deployment model.

The final decision is based on retrained validation/test performance, not feature importance alone.

In [5]:
print('Final model artifacts:')
for path in sorted(FINAL.rglob('*')):
    if path.is_file(): print(path.relative_to(FINAL))

Final model artifacts:
accuracy_first/feature_importance.csv
accuracy_first/metadata.json
accuracy_first/model.json
accuracy_first/shap_bar.png
accuracy_first/shap_summary.png
common_metrics.csv
feature_definitions.json
predictions.csv
research_preserving/feature_importance.csv
research_preserving/metadata.json
research_preserving/model.json
research_preserving/shap_bar.png
research_preserving/shap_summary.png
tuning_results.csv
